In [ ]:
!pip install wandb

In [ ]:
import numpy as np
import pandas as pd
import os
import random
import glob
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, accuracy_score
from kaggle_secrets import UserSecretsClient
import wandb
import shutil

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- Device Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Paths and Spectrogram Configuration ---
REAL_AUDIO_PATH = "/kaggle/input/the-lj-speech-dataset/LJSpeech-1.1/wavs"
FAKE_AUDIO_PATH = "/kaggle/input/wavefake-test/generated_audio"

SR = 16000
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
MAX_FRAMES_SPEC = 313 # Effective width for ViT patching will be 304 (19*16)
FMIN = 0.0
FMAX = None
APPLY_AUGMENTATION = True # Apply for training sets
NUM_TIME_MASKS = 1
NUM_FREQ_MASKS = 1
TIME_MASK_MAX_WIDTH = 40
FREQ_MASK_MAX_WIDTH = 15
NORM_EPSILON = 1e-6
MASK_REPLACEMENT_VALUE = 0.0
LIMIT_FILES = None # Set to a small number (e.g., 200) for quick testing, None for full dataset
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

# --- Training Hyperparameters (Common) ---
BATCH_SIZE = 32
NUM_WORKERS = 2 # os.cpu_count()
LEARNING_RATE = 1e-4
EPOCHS = 20 # As requested
WEIGHT_DECAY = 1e-4
PATIENCE_LIMIT = 7 # For early stopping

# --- Model-Specific Base Hyperparameters ---
VIT_PATCH_SIZE = 16
VIT_BASE_DROP_RATE = 0.1
VIT_BASE_ATTN_DROP_RATE = 0.1
CNN_BASE_DROPOUT_RATE = 0.4


# --- WandB Login with Kaggle Secrets ---
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api_key")
    wandb.login(key=wandb_api_key)
    print("WandB login successful using Kaggle Secrets.")
except Exception as e:
    print(f"Failed to login to WandB via Kaggle Secrets: {e}. Falling back to environment variable or manual login.")
    wandb.login()

# Note: wandb.init() will be called inside the loop for each model configuration.
# The initial top-level wandb.init() from the original script is removed.

In [ ]:
# --- 1. Data Loading and Preprocessing Functions ---
def get_audio_files_and_labels(
    real_dir, 
    fake_dir_parent, # Đổi tên để rõ là thư mục cha của các subdir fake
    limit_files=None,
    num_manual_test_samples=100,
    manual_export_base_dir="manual_dataset_exported" # Tên thư mục export mới
):
    # --- Lấy danh sách file Real và Fake ban đầu ---
    all_real_files = glob.glob(os.path.join(real_dir, '*.wav'))
    
    fake_subdirs = [os.path.join(fake_dir_parent, d) for d in os.listdir(fake_dir_parent)
                    if os.path.isdir(os.path.join(fake_dir_parent, d))]
    
    all_fake_files_pool = []
    for subdir in fake_subdirs:
        all_fake_files_pool.extend(glob.glob(os.path.join(subdir, '*.wav')))

    if not all_real_files:
        print("Warning: No real audio files found.")
        # return [], [] # Có thể thoát sớm nếu không có real
    if not all_fake_files_pool:
        print("Warning: No fake audio files found in subdirectories.")
        # return [], [] # Có thể thoát sớm nếu không có fake

    print(f"Initially found {len(all_real_files)} real files and {len(all_fake_files_pool)} total fake files.")

    # --- Xáo trộn danh sách file gốc TRƯỚC KHI tách ---
    random.shuffle(all_real_files)
    random.shuffle(all_fake_files_pool)

    # --- Tách Manual Test Set (100 Real, 100 Fake) ---
    if len(all_real_files) < num_manual_test_samples or len(all_fake_files_pool) < num_manual_test_samples:
        print(f"Warning: Not enough files to create a manual test set of {num_manual_test_samples} per class.")
        # Xử lý trường hợp này: có thể không tạo manual test, hoặc tạo với số lượng ít hơn
        # Hiện tại, sẽ cố gắng lấy tối đa có thể cho manual test, phần còn lại vẫn dùng cho train
        actual_manual_real_count = min(len(all_real_files), num_manual_test_samples)
        actual_manual_fake_count = min(len(all_fake_files_pool), num_manual_test_samples)
    else:
        actual_manual_real_count = num_manual_test_samples
        actual_manual_fake_count = num_manual_test_samples

    manual_test_real_files = all_real_files[:actual_manual_real_count]
    remaining_real_files = all_real_files[actual_manual_real_count:]

    manual_test_fake_files = all_fake_files_pool[:actual_manual_fake_count]
    remaining_fake_files_pool = all_fake_files_pool[actual_manual_fake_count:]
    
    print(f"Separated {len(manual_test_real_files)} real and {len(manual_test_fake_files)} fake files for manual testing.")

    # Export manual test set
    if manual_export_base_dir:
        real_export_dir = os.path.join(manual_export_base_dir, 'real')
        fake_export_dir = os.path.join(manual_export_base_dir, 'fake')
        os.makedirs(real_export_dir, exist_ok=True)
        os.makedirs(fake_export_dir, exist_ok=True)

        for file_path in manual_test_real_files:
            try:
                shutil.copy(file_path, os.path.join(real_export_dir, os.path.basename(file_path)))
            except Exception as e:
                print(f"Failed to copy {file_path} to manual real dir: {e}")
        for file_path in manual_test_fake_files:
            try:
                shutil.copy(file_path, os.path.join(fake_export_dir, os.path.basename(file_path)))
            except Exception as e:
                print(f"Failed to copy {file_path} to manual fake dir: {e}")
        print(f"Manual test files exported to {manual_export_base_dir}")
        
    # --- Sử dụng PHẦN CÒN LẠI cho training/validation/auto-test ---
    # Mục tiêu là cân bằng số lượng fake với số lượng real còn lại
    target_fake_for_main = len(remaining_real_files)
    
    selected_fake_for_main = []
    if not remaining_fake_files_pool:
        print("Warning: No fake files left for main training after manual test separation.")
    elif len(remaining_fake_files_pool) > target_fake_for_main:
        selected_fake_for_main = random.sample(remaining_fake_files_pool, target_fake_for_main)
    else:
        selected_fake_for_main = list(remaining_fake_files_pool) # Lấy hết phần còn lại
        
    print(f"Using {len(remaining_real_files)} real audio files for main process.")
    print(f"Using {len(selected_fake_for_main)} fake audio files for main process (sampled from {len(remaining_fake_files_pool)} remaining fake).")

    # Đây là dữ liệu sẽ được dùng để chia train/val/test tự động
    current_real_files = remaining_real_files
    current_fake_files = selected_fake_for_main

    if not current_real_files and not current_fake_files: # Kiểm tra nếu cả hai đều rỗng
        if limit_files is None or limit_files == 0 : # Chỉ in nếu không phải do limit_files
             print("Warning: No real or fake files available for main process after selections.")
    # Các kiểm tra lỗi khác có thể giữ nguyên hoặc điều chỉnh
    
    # Xử lý limit_files cho phần dữ liệu chính này
    if limit_files:
        print(f"Limiting files for main process. Attempting to sample up to {limit_files // 2} from each class.")
        current_real_files = random.sample(current_real_files, min(limit_files // 2, len(current_real_files)))
        current_fake_files = random.sample(current_fake_files, min(limit_files // 2, len(current_fake_files)))
        print(f"Selected {len(current_real_files)} real and {len(current_fake_files)} fake files for main process after limiting.")

    filepaths = current_real_files + current_fake_files
    labels = [0] * len(current_real_files) + [1] * len(current_fake_files)

    if not filepaths:
        print("No files selected for main process to return.")
        return [], [] # Trả về rỗng nếu không có file nào

    combined = list(zip(filepaths, labels))
    random.shuffle(combined) # Xáo trộn lần cuối
    filepaths_shuffled, labels_shuffled = [], []
    if combined:
        filepaths_shuffled, labels_shuffled = zip(*combined)
    
    print(f"Total files returned for main process: {len(filepaths_shuffled)} "
          f"(Real: {labels_shuffled.count(0)}, Fake: {labels_shuffled.count(1)})")
    return list(filepaths_shuffled), list(labels_shuffled)

def audio_to_melspectrogram(filepath, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, max_frames=MAX_FRAMES_SPEC, fmin=FMIN, fmax=FMAX):
    try:
        y, sr_orig = librosa.load(filepath, sr=None)
        if sr_orig != sr: y = librosa.resample(y, orig_sr=sr_orig, target_sr=sr)
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, fmin=fmin, fmax=fmax if fmax is not None else sr/2)
        log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
        current_frames = log_mel_spectrogram.shape[1]
        if current_frames < max_frames:
            pad_value = log_mel_spectrogram.min() # Pad with min value
            pad_width = max_frames - current_frames
            padded_log_mel_spectrogram = np.pad(log_mel_spectrogram, ((0, 0), (0, pad_width)), mode='constant', constant_values=pad_value)
            return padded_log_mel_spectrogram
        elif current_frames > max_frames:
            truncated_log_mel_spectrogram = log_mel_spectrogram[:, :max_frames]
            return truncated_log_mel_spectrogram
        else: return log_mel_spectrogram
    except Exception as e:
        # print(f"Error processing {filepath}: {e}") # Optional: for debugging
        return None

In [ ]:
# --- 2. PyTorch Dataset ---
class AudioDataset(Dataset):
    def __init__(self, filepaths, labels, transform_spectrogram_fn, augment=False, is_vit_input=False,
                 time_mask_max_width=TIME_MASK_MAX_WIDTH, freq_mask_max_width=FREQ_MASK_MAX_WIDTH,
                 num_time_masks=NUM_TIME_MASKS, num_freq_masks=NUM_FREQ_MASKS,
                 mask_replacement_value=MASK_REPLACEMENT_VALUE):
        self.filepaths = filepaths
        self.labels = labels
        self.transform_spectrogram_fn = transform_spectrogram_fn
        self.augment = augment
        self.is_vit_input = is_vit_input # True for ViT (3 channels), False for CNN (1 channel)
        self.time_mask_max_width = time_mask_max_width
        self.freq_mask_max_width = freq_mask_max_width
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks
        self.mask_replacement_value = mask_replacement_value

    def __len__(self):
        return len(self.filepaths)

    def _apply_time_mask(self, spectrogram):
        augmented_spec = np.copy(spectrogram)
        num_frames = augmented_spec.shape[1]
        for _ in range(self.num_time_masks):
            if self.time_mask_max_width > 0 and num_frames > self.time_mask_max_width:
                t = random.randint(1, self.time_mask_max_width)
                t0 = random.randint(0, num_frames - t)
                augmented_spec[:, t0:t0 + t] = self.mask_replacement_value
        return augmented_spec

    def _apply_freq_mask(self, spectrogram):
        augmented_spec = np.copy(spectrogram)
        num_mels = augmented_spec.shape[0]
        for _ in range(self.num_freq_masks):
            if self.freq_mask_max_width > 0 and num_mels > self.freq_mask_max_width:
                f = random.randint(1, self.freq_mask_max_width)
                f0 = random.randint(0, num_mels - f)
                augmented_spec[f0:f0 + f, :] = self.mask_replacement_value
        return augmented_spec

    def __getitem__(self, idx):
        filepath = self.filepaths[idx]
        label = self.labels[idx]
        mel_spec = self.transform_spectrogram_fn(filepath)
        if mel_spec is None: return None # Handle cases where audio_to_melspectrogram fails

        if self.augment:
            mel_spec = self._apply_time_mask(mel_spec)
            mel_spec = self._apply_freq_mask(mel_spec)

        # Normalize: Z-score normalization per spectrogram
        mean = np.mean(mel_spec)
        std = np.std(mel_spec)
        mel_spec_normalized = (mel_spec - mean) / (std + NORM_EPSILON)

        if self.is_vit_input:
            # Stack to 3 channels for ViT
            mel_spec_final = np.stack([mel_spec_normalized]*3, axis=0)
        else:
            # Add channel dimension for CNN
            mel_spec_final = np.expand_dims(mel_spec_normalized, axis=0)

        mel_spec_tensor = torch.tensor(mel_spec_final, dtype=torch.float32)
        label_tensor = torch.tensor(label, dtype=torch.float32) # BCEWithLogitsLoss expects float
        return mel_spec_tensor, label_tensor

def collate_fn_skip_none_vit(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch:
        # Return empty tensors with correct shape if batch is empty after filtering
        return torch.empty((0, 3, N_MELS, MAX_FRAMES_SPEC)), torch.empty((0,))
    return torch.utils.data.dataloader.default_collate(batch)

def collate_fn_skip_none_cnn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if not batch:
        # Return empty tensors with correct shape
        return torch.empty((0, 1, N_MELS, MAX_FRAMES_SPEC)), torch.empty((0,))
    return torch.utils.data.dataloader.default_collate(batch)

In [ ]:
# --- 3. Data Splitting ---
filepaths_all, labels_all = get_audio_files_and_labels(REAL_AUDIO_PATH, FAKE_AUDIO_PATH, limit_files=LIMIT_FILES)

if not filepaths_all:
    raise ValueError("Halting: No audio files found or loaded. Check paths and limit_files settings.")

print(f"\nTotal samples before splitting: {len(filepaths_all)} (Labels: Real={labels_all.count(0)}, Fake={labels_all.count(1)})")

# Ensure there's enough data for splitting and stratification
if len(filepaths_all) < 10 : # Arbitrary small number, adjust if necessary
    print("Warning: Very few samples available. Splitting might result in empty sets or issues with stratification.")
    # Handle small dataset case: maybe use all for training, or a different split strategy.
    # For now, proceed but be aware of potential issues.

# Stratification requires at least 2 members for each class in the set being split.
can_stratify_all = len(set(labels_all)) > 1 and all(labels_all.count(l) >= 2 for l in set(labels_all))

X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    filepaths_all, labels_all,
    test_size=(VALIDATION_RATIO + TEST_RATIO), # Combined size for validation and test
    random_state=SEED,
    stratify=labels_all if can_stratify_all else None
)

if X_temp_paths: # If there are samples for validation/test
    relative_test_ratio = TEST_RATIO / (VALIDATION_RATIO + TEST_RATIO)
    can_stratify_temp = len(set(y_temp)) > 1 and all(y_temp.count(l) >= 2 for l in set(y_temp))
    if len(X_temp_paths) < 2 : # Cannot split further if only one sample
         X_val_paths, y_val = X_temp_paths, y_temp
         X_test_paths, y_test = [], []
    else:
        X_val_paths, X_test_paths, y_val, y_test = train_test_split(
            X_temp_paths, y_temp,
            test_size=relative_test_ratio,
            random_state=SEED,
            stratify=y_temp if can_stratify_temp else None
        )
else: # No samples left for validation/test
    X_val_paths, X_test_paths, y_val, y_test = [], [], [], []

print(f"Training samples: {len(X_train_paths)} (Real={y_train.count(0)}, Fake={y_train.count(1)})")
print(f"Validation samples: {len(X_val_paths)} (Real={y_val.count(0)}, Fake={y_val.count(1)})")
print(f"Test samples: {len(X_test_paths)} (Real={y_test.count(0)}, Fake={y_test.count(1)})")

# Log dataset statistics to a temporary WandB run if needed for global overview, or log per run.
# For now, these stats will be logged for each individual model run via its own wandb.init() config.

In [ ]:
# --- 4. PyTorch Datasets and DataLoaders for ViT and CNN (Create once) ---

# ViT Datasets & DataLoaders
train_dataset_vit = AudioDataset(
    X_train_paths, y_train, transform_spectrogram_fn=audio_to_melspectrogram,
    augment=APPLY_AUGMENTATION, is_vit_input=True
)
val_dataset_vit = AudioDataset(
    X_val_paths, y_val, transform_spectrogram_fn=audio_to_melspectrogram,
    augment=False, is_vit_input=True # No augmentation for validation
)
train_loader_vit = DataLoader(
    train_dataset_vit, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_vit
)
val_loader_vit = DataLoader(
    val_dataset_vit, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_vit
)
test_loader_vit = None
if X_test_paths:
    test_dataset_vit = AudioDataset(
        X_test_paths, y_test, transform_spectrogram_fn=audio_to_melspectrogram,
        augment=False, is_vit_input=True # No augmentation for test
    )
    test_loader_vit = DataLoader(
        test_dataset_vit, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_vit
    )
else:
    print("Test set is empty, test_loader_vit not created.")

# CNN Datasets & DataLoaders
train_dataset_cnn = AudioDataset(
    X_train_paths, y_train, transform_spectrogram_fn=audio_to_melspectrogram,
    augment=APPLY_AUGMENTATION, is_vit_input=False
)
val_dataset_cnn = AudioDataset(
    X_val_paths, y_val, transform_spectrogram_fn=audio_to_melspectrogram,
    augment=False, is_vit_input=False # No augmentation for validation
)
train_loader_cnn = DataLoader(
    train_dataset_cnn, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_cnn
)
val_loader_cnn = DataLoader(
    val_dataset_cnn, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_cnn
)
test_loader_cnn = None
if X_test_paths:
    test_dataset_cnn = AudioDataset(
        X_test_paths, y_test, transform_spectrogram_fn=audio_to_melspectrogram,
        augment=False, is_vit_input=False # No augmentation for test
    )
    test_loader_cnn = DataLoader(
        test_dataset_cnn, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_skip_none_cnn
    )
else:
    print("Test set is empty, test_loader_cnn not created.")

# Quick test of DataLoaders (optional, can be commented out for final runs)
# This will be logged per-run if sample spectrograms are logged.
# The original script's detailed DataLoader test section is omitted here for brevity,
# as similar logging will happen inside the main training loop per model.

In [ ]:
# --- 5. PyTorch Vision Transformer (ViT) Model ---
class PatchEmbed(nn.Module):
    def __init__(self, img_size=(N_MELS, MAX_FRAMES_SPEC), patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        # Calculate grid size based on floor division, compatible with Conv2d behavior
        self.grid_size = (img_size[0] // patch_size, img_size[1] // patch_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        # Input image size might be slightly different if MAX_FRAMES_SPEC is not a multiple of patch_size
        # The Conv2d will handle this by effectively processing the largest multiple.
        # assert H == self.img_size[0] and W == self.img_size[1], \
        #     f"Input image size ({H}*{W}) doesn't match model expected input size ({self.img_size[0]}*{self.img_size[1]})."
        # If H or W are not exact multiples, Conv2d output size will be floor(dim/patch_size).
        # Example: H=128, patch_size=16 -> H_out=8. W=313, patch_size=16 -> W_out=19.
        # This matches self.grid_size calculation.
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x) # Dropout after first activation
        x = self.fc2(x)
        x = self.drop(x) # Dropout after second linear
        return x

class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, img_size=(N_MELS, MAX_FRAMES_SPEC), patch_size=16, in_chans=3, num_classes=1,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4., qkv_bias=True,
                 drop_rate=0., attn_drop_rate=0.): # Default drop rates to 0, will be set by constants
        super().__init__()
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim)) # +1 for cls_token
        self.pos_drop = nn.Dropout(p=drop_rate)

        self.blocks = nn.ModuleList([
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias,
                  drop=drop_rate, attn_drop=attn_drop_rate)
            for i in range(depth)])
        self.norm = nn.LayerNorm(embed_dim) # Final LayerNorm

        # Classifier head
        self.head = nn.Linear(embed_dim, num_classes)

        # Weight init
        nn.init.trunc_normal_(self.pos_embed, std=.02)
        nn.init.trunc_normal_(self.cls_token, std=.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_features(self, x):
        B = x.shape[0]
        x = self.patch_embed(x) # (B, num_patches, embed_dim)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat((cls_tokens, x), dim=1) # (B, num_patches+1, embed_dim)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        return x[:, 0] # Return CLS token

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x

In [ ]:
# --- 6. PyTorch CNN Model (Modified for Parameterization) ---
class AudioCNN(nn.Module):
    def __init__(self, num_classes=1, dropout_rate=0.4,
                 channels_list=None, fc_nodes_list=None,
                 n_mels=N_MELS, max_frames_spec=MAX_FRAMES_SPEC):
        super(AudioCNN, self).__init__()

        if channels_list is None: # Default to a "Large" like configuration
            channels_list = [32, 64, 128, 256]
        if fc_nodes_list is None: # Default to a "Large" like configuration
            fc_nodes_list = [512, 128]

        self.conv_layers = nn.ModuleList()
        self.bn_conv_layers = nn.ModuleList()
        self.pool_layers = nn.ModuleList()
        self.drop_conv_layers = nn.ModuleList()

        in_channels = 1 # Input is single channel (grayscale spectrogram)
        for i, out_channels in enumerate(channels_list):
            self.conv_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
            self.bn_conv_layers.append(nn.BatchNorm2d(out_channels))
            self.pool_layers.append(nn.MaxPool2d(kernel_size=2)) # Halves dimensions
            # Original dropout was dropout_rate/2 for first two, dropout_rate for last two conv blocks
            current_dropout_rate = dropout_rate / 2 if i < len(channels_list) / 2 else dropout_rate
            self.drop_conv_layers.append(nn.Dropout2d(current_dropout_rate))
            in_channels = out_channels

        # Calculate the size of the flattened features after conv and pool layers
        num_pool_operations = len(channels_list) # Assuming one pool per conv block
        height_after_convs = n_mels // (2**num_pool_operations)
        width_after_convs = max_frames_spec // (2**num_pool_operations)
        
        fc_in_features = channels_list[-1] * height_after_convs * width_after_convs

        self.fc_layers = nn.ModuleList()
        self.bn_fc_layers = nn.ModuleList()
        self.drop_fc_layers = nn.ModuleList()

        current_fc_in_dim = fc_in_features
        for i, fc_out_dim in enumerate(fc_nodes_list):
            self.fc_layers.append(nn.Linear(current_fc_in_dim, fc_out_dim))
            self.bn_fc_layers.append(nn.BatchNorm1d(fc_out_dim))
            self.drop_fc_layers.append(nn.Dropout(dropout_rate)) # Consistent dropout for all FC layers
            current_fc_in_dim = fc_out_dim
        
        self.output_fc = nn.Linear(current_fc_in_dim, num_classes) # Final classification layer

    def forward(self, x):
        # Convolutional blocks
        for i in range(len(self.conv_layers)):
            x = self.conv_layers[i](x)
            x = self.bn_conv_layers[i](x)
            x = F.relu(x)
            x = self.pool_layers[i](x)
            x = self.drop_conv_layers[i](x)
        
        # Flatten for FC layers
        x = x.view(x.size(0), -1)
        
        # Fully connected blocks
        for i in range(len(self.fc_layers)):
            x = self.fc_layers[i](x)
            x = self.bn_fc_layers[i](x)
            x = F.relu(x)
            x = self.drop_fc_layers[i](x)
            
        # Output layer
        x = self.output_fc(x)
        return x

In [ ]:
# --- 7. Training Loop and Evaluation Function ---
def train_one_epoch(model, train_loader, criterion, optimizer, device, epoch_num, num_epochs, model_name="Model"):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch_num+1}/{num_epochs} [{model_name} Training]", unit="batch", leave=False)
    for inputs, labels in progress_bar:
        if inputs.nelement() == 0: continue # Skip if batch is empty after collate_fn filtering
        inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1) # Ensure labels are [B, 1] for BCEWithLogitsLoss

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = torch.sigmoid(outputs) > 0.5
        correct_predictions += (preds == labels).sum().item()
        total_samples += labels.size(0)
        
        if total_samples > 0:
            progress_bar.set_postfix(loss=loss.item(), acc=correct_predictions/total_samples)
        else:
            progress_bar.set_postfix(loss=loss.item(), acc=0)
            
    epoch_loss = running_loss / total_samples if total_samples > 0 else 0
    epoch_acc = correct_predictions / total_samples if total_samples > 0 else 0
    return epoch_loss, epoch_acc

def evaluate_model_pytorch(model, val_loader, criterion, device, epoch_num=None, num_epochs=None, model_name="Model", is_test_set=False):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_labels_true = []
    all_preds_probs = []
    
    desc_str = f"{model_name} Evaluating"
    if not is_test_set and epoch_num is not None and num_epochs is not None:
        desc_str = f"Epoch {epoch_num+1}/{num_epochs} [{model_name} Validation]"
    elif is_test_set:
        desc_str = f"[{model_name} Test Set Evaluation]"

    progress_bar = tqdm(val_loader, desc=desc_str, unit="batch", leave=False)
    with torch.no_grad():
        for inputs, labels in progress_bar:
            if inputs.nelement() == 0: continue
            inputs, labels_true_batch = inputs.to(device), labels.to(device).unsqueeze(1)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels_true_batch) # Can be None if criterion is not passed for test set without loss
            
            if loss is not None:
                running_loss += loss.item() * inputs.size(0)
            
            probs = torch.sigmoid(outputs)
            preds_binary = probs > 0.5
            
            correct_predictions += (preds_binary == labels_true_batch).sum().item()
            total_samples += labels_true_batch.size(0)
            
            all_labels_true.extend(labels_true_batch.cpu().numpy().flatten())
            all_preds_probs.extend(probs.cpu().numpy().flatten())

            current_acc = correct_predictions/total_samples if total_samples > 0 else 0
            if loss is not None:
                progress_bar.set_postfix(loss=loss.item(), acc=current_acc)
            else:
                progress_bar.set_postfix(acc=current_acc)

    epoch_loss = running_loss / total_samples if total_samples > 0 else float('inf') # Or 0 if criterion was None
    epoch_acc = correct_predictions / total_samples if total_samples > 0 else 0
    
    return epoch_loss, epoch_acc, np.array(all_labels_true), np.array(all_preds_probs)

def plot_history(train_losses, val_losses, train_accs, val_accs, model_name_display, filename_base):
    epochs_range = range(1, len(train_losses) + 1)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, train_losses, label='Training Loss')
    plt.plot(epochs_range, val_losses, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title(f'{model_name_display} - Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, train_accs, label='Training Accuracy')
    plt.plot(epochs_range, val_accs, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title(f'{model_name_display} - Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')

    plt.tight_layout()
    save_path = f"training_history_{filename_base}.png"
    plt.savefig(save_path)
    wandb.log({f"training_history_plot": wandb.Image(save_path)}) # Log with a generic key, name comes from run
    plt.close()

In [ ]:
# --- 8. Model Configurations ---
model_configurations = [
    # ViT Models
    {"model_type": "ViT", "size": "Small", "params": {"embed_dim": 192, "depth": 5, "num_heads": 6, "mlp_ratio": 4.0}, "is_vit_input": True},
    {"model_type": "ViT", "size": "Medium", "params": {"embed_dim": 384, "depth": 6, "num_heads": 6, "mlp_ratio": 4.0}, "is_vit_input": True},
    {"model_type": "ViT", "size": "Large", "params": {"embed_dim": 512, "depth": 6, "num_heads": 8, "mlp_ratio": 4.0}, "is_vit_input": True},
    # CNN Models
    {"model_type": "CNN", "size": "Small", "params": {"channels": [16, 32, 64, 128], "fc_nodes": [128, 32]}, "is_vit_input": False},
    {"model_type": "CNN", "size": "Medium", "params": {"channels": [32, 64, 128, 256], "fc_nodes": [256, 128]}, "is_vit_input": False},
    {"model_type": "CNN", "size": "Large", "params": {"channels": [32, 64, 128, 256], "fc_nodes": [512, 128]}, "is_vit_input": False},
]

# --- 9. Main Training and Evaluation Loop ---
common_criterion = nn.BCEWithLogitsLoss()

# If N_MELS or MAX_FRAMES_SPEC are not divisible by ViT patch size, print a warning once.
if N_MELS % VIT_PATCH_SIZE != 0 or MAX_FRAMES_SPEC % VIT_PATCH_SIZE != 0:
    eff_H_vit = (N_MELS // VIT_PATCH_SIZE) * VIT_PATCH_SIZE
    eff_W_vit = (MAX_FRAMES_SPEC // VIT_PATCH_SIZE) * VIT_PATCH_SIZE
    print(f"Global Warning: For ViT models with patch_size={VIT_PATCH_SIZE}:")
    print(f"  N_MELS ({N_MELS}) or MAX_FRAMES_SPEC ({MAX_FRAMES_SPEC}) may not be perfectly divisible.")
    print(f"  Effective input to PatchEmbed will be ({eff_H_vit}, {eff_W_vit}).")


for config_idx, config in enumerate(model_configurations):
    model_type = config["model_type"]
    model_size = config["size"]
    model_specific_params = config["params"]
    is_vit_model = config["is_vit_input"]

    run_name = f"{time.strftime('%Y%m%d_%H%M%S')}_{model_type}_{model_size}"
    wandb.init(
        project="ASM01_DAT301m_MultiModel", # Using a distinct project name for these runs
        name=run_name,
        config={
            # Common hyperparameters
            "learning_rate": LEARNING_RATE,
            "epochs_max": EPOCHS,
            "batch_size": BATCH_SIZE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "optimizer": "AdamW",
            # Data params
            "sr": SR, "n_fft": N_FFT, "hop_length": HOP_LENGTH, "n_mels": N_MELS,
            "max_frames_spec": MAX_FRAMES_SPEC, "fmin": FMIN, "fmax": FMAX,
            "apply_augmentation": APPLY_AUGMENTATION,
            "num_time_masks": NUM_TIME_MASKS, "num_freq_masks": NUM_FREQ_MASKS,
            "time_mask_max_width": TIME_MASK_MAX_WIDTH, "freq_mask_max_width": FREQ_MASK_MAX_WIDTH,
            # Dataset split info
            "train_samples_count": len(X_train_paths),
            "val_samples_count": len(X_val_paths),
            "test_samples_count": len(X_test_paths),
            # Model specific architectural params
            "model_type": model_type,
            "model_size": model_size,
            **model_specific_params # Adds e.g., embed_dim, depth for ViT; channels, fc_nodes for CNN
        }
    )

    # Log specific base rates if they are part of the run's identity
    if model_type == "ViT":
        wandb.config.update({
            "vit_patch_size": VIT_PATCH_SIZE,
            "vit_drop_rate": VIT_BASE_DROP_RATE,
            "vit_attn_drop_rate": VIT_BASE_ATTN_DROP_RATE,
        })
    elif model_type == "CNN":
        wandb.config.update({
            "cnn_dropout_rate": CNN_BASE_DROPOUT_RATE,
        })

    print(f"\n--- [{config_idx+1}/{len(model_configurations)}] Processing: {model_type} ({model_size}) ---")

    # 1. Instantiate Model
    if model_type == "ViT":
        model = VisionTransformer(
            img_size=(N_MELS, MAX_FRAMES_SPEC),
            patch_size=VIT_PATCH_SIZE,
            in_chans=3, num_classes=1, # ViT expects 3 channels, binary classification
            embed_dim=model_specific_params["embed_dim"],
            depth=model_specific_params["depth"],
            num_heads=model_specific_params["num_heads"],
            mlp_ratio=model_specific_params["mlp_ratio"],
            qkv_bias=True, # Standard ViT choice
            drop_rate=VIT_BASE_DROP_RATE,
            attn_drop_rate=VIT_BASE_ATTN_DROP_RATE
        ).to(DEVICE)
        current_train_loader = train_loader_vit
        current_val_loader = val_loader_vit
        current_test_loader = test_loader_vit
    else: # CNN
        model = AudioCNN(
            num_classes=1, # Binary classification
            dropout_rate=CNN_BASE_DROPOUT_RATE,
            channels_list=model_specific_params["channels"],
            fc_nodes_list=model_specific_params["fc_nodes"],
            n_mels=N_MELS,
            max_frames_spec=MAX_FRAMES_SPEC
        ).to(DEVICE)
        current_train_loader = train_loader_cnn
        current_val_loader = val_loader_cnn
        current_test_loader = test_loader_cnn

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model: {model_type} ({model_size}), Trainable Parameters: {total_params:,}")
    wandb.log({"total_trainable_parameters": total_params})
    
    # Log a sample spectrogram based on model type (once per run)
    # Check if dataloader is not None and has data
    sample_loader_for_img = current_train_loader
    if sample_loader_for_img and len(sample_loader_for_img) > 0:
        try:
            sample_batch_x, sample_batch_y = next(iter(sample_loader_for_img))
            if sample_batch_x.nelement() > 0:
                plt.figure(figsize=(10, 4))
                # For ViT, input is (B,3,H,W), for CNN it's (B,1,H,W). Take first channel for display.
                img_to_show = sample_batch_x[0, 0, :, :].cpu().numpy()
                librosa.display.specshow(img_to_show, sr=SR, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel')
                plt.colorbar(format='%+2.0f dB')
                plt.title(f'Sample Input Spectrogram ({model_type} {model_size}, Label: {sample_batch_y[0].item():.0f})')
                plt.tight_layout()
                img_path = f"sample_spectrogram_{model_type.lower()}_{model_size.lower()}.png"
                plt.savefig(img_path)
                wandb.log({"sample_input_spectrogram": wandb.Image(img_path)})
                plt.close()
            else: print(f"  Skipping sample spectrogram: First batch from {model_type} loader is empty.")
        except StopIteration: print(f"  Skipping sample spectrogram: {model_type} Train loader is empty.")
        except Exception as e_img: print(f"  Error logging sample spectrogram: {e_img}")
    else: print(f"  Skipping sample spectrogram: {model_type} Train loader is unavailable or empty.")


    # 2. Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # 3. Training Loop
    history_train_loss, history_val_loss = [], []
    history_train_acc, history_val_acc = [], []
    best_val_loss_for_model = float('inf')
    best_epoch_for_model = -1
    patience_counter_for_model = 0
    
    model_save_path = f'best_{model_type.lower()}_{model_size.lower()}_model.pth'

    print(f"  Starting training for {model_type} ({model_size}) on {DEVICE}...")
    start_time_total_train = time.time()

    for epoch in range(EPOCHS):
        epoch_start_time = time.time()
        
        train_loss, train_acc = train_one_epoch(
            model, current_train_loader, common_criterion, optimizer, DEVICE,
            epoch, EPOCHS, model_name=f"{model_type} {model_size}"
        )
        val_loss, val_acc, _, _ = evaluate_model_pytorch(
            model, current_val_loader, common_criterion, DEVICE,
            epoch, EPOCHS, model_name=f"{model_type} {model_size}"
        )
        epoch_duration = time.time() - epoch_start_time

        print(f"    Epoch {epoch+1}/{EPOCHS} - {model_type} {model_size} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} | "
              f"Duration: {epoch_duration:.2f}s")
        
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss, "train_accuracy": train_acc,
            "val_loss": val_loss, "val_accuracy": val_acc,
            "epoch_duration_seconds": epoch_duration
        })

        history_train_loss.append(train_loss)
        history_val_loss.append(val_loss)
        history_train_acc.append(train_acc)
        history_val_acc.append(val_acc)

        if val_loss < best_val_loss_for_model:
            best_val_loss_for_model = val_loss
            best_epoch_for_model = epoch + 1
            torch.save(model.state_dict(), model_save_path)
            print(f"      Epoch {epoch+1}: Val loss improved to {val_loss:.4f}. Model saved to {model_save_path}")
            patience_counter_for_model = 0
        else:
            patience_counter_for_model += 1
            print(f"      Epoch {epoch+1}: Val loss ({val_loss:.4f}) did not improve from {best_val_loss_for_model:.4f}. Patience: {patience_counter_for_model}/{PATIENCE_LIMIT}")
        
        if patience_counter_for_model >= PATIENCE_LIMIT:
            print(f"    Early stopping triggered at epoch {epoch+1} for {model_type} {model_size}.")
            wandb.log({"early_stopping_epoch": epoch + 1})
            break
    
    total_training_time_model = time.time() - start_time_total_train
    print(f"  --- {model_type} ({model_size}) Training Finished ---")
    print(f"  Total Training Time: {total_training_time_model // 60:.0f}m {total_training_time_model % 60:.0f}s")
    print(f"  Best validation loss: {best_val_loss_for_model:.4f} at epoch {best_epoch_for_model}")
    
    wandb.log({
        "total_training_time_minutes": total_training_time_model / 60,
        "best_val_loss": best_val_loss_for_model,
        "best_epoch": best_epoch_for_model,
        "completed_epochs": epoch + 1 # Actual epochs run
    })
    # Save the model file as a W&B artifact at the end of training
    if os.path.exists(model_save_path):
        model_artifact = wandb.Artifact(f"{model_type.lower()}_{model_size.lower()}_model", type="model")
        model_artifact.add_file(model_save_path)
        wandb.log_artifact(model_artifact)


    # 4. Plot Training History
    plot_filename_base = f"{model_type.lower()}_{model_size.lower()}"
    plot_history(history_train_loss, history_val_loss, history_train_acc, history_val_acc,
                 model_name_display=f"{model_type} {model_size}", filename_base=plot_filename_base)

    # 5. Test Set Evaluation
    if current_test_loader and os.path.exists(model_save_path):
        print(f"\n  --- Evaluating {model_type} ({model_size}) on Test Set ---")
        
        # Re-instantiate the model structure and load the best weights
        if model_type == "ViT":
            test_model = VisionTransformer(
                img_size=(N_MELS, MAX_FRAMES_SPEC), patch_size=VIT_PATCH_SIZE, in_chans=3, num_classes=1,
                embed_dim=model_specific_params["embed_dim"], depth=model_specific_params["depth"],
                num_heads=model_specific_params["num_heads"], mlp_ratio=model_specific_params["mlp_ratio"],
                qkv_bias=True, drop_rate=VIT_BASE_DROP_RATE, attn_drop_rate=VIT_BASE_ATTN_DROP_RATE
            ).to(DEVICE)
        else: # CNN
            test_model = AudioCNN(
                num_classes=1, dropout_rate=CNN_BASE_DROPOUT_RATE,
                channels_list=model_specific_params["channels"], fc_nodes_list=model_specific_params["fc_nodes"],
                n_mels=N_MELS, max_frames_spec=MAX_FRAMES_SPEC
            ).to(DEVICE)
        
        try:
            test_model.load_state_dict(torch.load(model_save_path, map_location=DEVICE))
            print(f"    Best {model_type} ({model_size}) model weights loaded from {model_save_path}")
            
            # Evaluate (criterion can be common_criterion or None if loss is not needed for test)
            test_loss, test_acc, test_labels_true, test_preds_probs = evaluate_model_pytorch(
                test_model, current_test_loader, common_criterion, DEVICE,
                model_name=f"{model_type} {model_size}", is_test_set=True
            )
            print(f"    {model_type} ({model_size}) Test Set - Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}")
            wandb.log({
                "test_loss": test_loss,
                "test_accuracy": test_acc
            })

            if len(test_labels_true) > 0 and len(test_preds_probs) > 0:
                test_preds_binary = (test_preds_probs > 0.5).astype(int)
                
                # Classification Report
                report_dict = classification_report(test_labels_true, test_preds_binary, target_names=['Real (0)', 'Fake (1)'], output_dict=True, zero_division=0)
                report_str = classification_report(test_labels_true, test_preds_binary, target_names=['Real (0)', 'Fake (1)'], zero_division=0)
                print("\n    Classification Report (Test Set):")
                print(report_str)
                wandb.log({
                    "test_cls_report_str": report_str, # Log as text
                    "test_precision_real": report_dict['Real (0)']['precision'],
                    "test_recall_real": report_dict['Real (0)']['recall'],
                    "test_f1_real": report_dict['Real (0)']['f1-score'],
                    "test_precision_fake": report_dict['Fake (1)']['precision'],
                    "test_recall_fake": report_dict['Fake (1)']['recall'],
                    "test_f1_fake": report_dict['Fake (1)']['f1-score'],
                    "test_macro_avg_precision": report_dict['macro avg']['precision'],
                    "test_macro_avg_recall": report_dict['macro avg']['recall'],
                    "test_macro_avg_f1": report_dict['macro avg']['f1-score'],
                    "test_weighted_avg_precision": report_dict['weighted avg']['precision'],
                    "test_weighted_avg_recall": report_dict['weighted avg']['recall'],
                    "test_weighted_avg_f1": report_dict['weighted avg']['f1-score']
                })

                # ROC AUC Score
                try:
                    if len(np.unique(test_labels_true)) > 1: # ROC AUC requires more than 1 class in y_true
                        roc_auc = roc_auc_score(test_labels_true, test_preds_probs)
                        print(f"    ROC AUC Score (Test Set): {roc_auc:.4f}")
                        wandb.log({"test_roc_auc": roc_auc})
                    else:
                        print("    ROC AUC Score: Not calculated (only one class present in test labels).")
                        wandb.log({"test_roc_auc": float('nan')}) # Log NaN or skip
                except ValueError as e_roc:
                    print(f"    Could not calculate ROC AUC: {e_roc}")
                    wandb.log({"test_roc_auc": float('nan')})

                # Confusion Matrix
                cm = confusion_matrix(test_labels_true, test_preds_binary)
                disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Real', 'Fake'])
                disp.plot(cmap=plt.cm.Blues)
                cm_title = f'Confusion Matrix - {model_type} {model_size} (Test)'
                plt.title(cm_title)
                cm_filename = f"confusion_matrix_{model_type.lower()}_{model_size.lower()}_test.png"
                plt.savefig(cm_filename)
                plt.close()
                wandb.log({"test_confusion_matrix": wandb.Image(cm_filename)})
            else:
                print("    Not enough data in test results for classification report/matrix.")
        except FileNotFoundError:
            print(f"    Error: Model file '{model_save_path}' not found for test evaluation.")
        except Exception as e_test:
            print(f"    An error occurred during {model_type} ({model_size}) test set evaluation: {e_test}")
    elif not current_test_loader:
        print(f"\n  {model_type} ({model_size}) Test loader is not available. Skipping test set evaluation.")
    elif not os.path.exists(model_save_path):
        print(f"\n  Best model file {model_save_path} not found. Skipping test set evaluation for {model_type} {model_size}.")

    wandb.finish() # End of W&B run for this model configuration

print("\n--- All model configurations processed ---")